# 13 - Reranker Top-N Sweep

Investigation Task 3 (project-owner-approved menu item). Question: is `rerank_top_n_blocks=8`
(the shipped default) too aggressive for this boilerplate-heavy financial corpus, or would a
larger N (12/16/20/24) do meaningfully better -- or does N barely matter at all?

**Cheap by construction:** `CohereReranker.score_blocks()` scores every candidate block against
the query in ONE Bedrock Rerank call regardless of what top-N cutoff gets applied afterward
(billing is per-100-chunks-submitted, not per-chunk-kept). So this sweep costs exactly the same
single rerank pass as the N=8 config already shipped -- no repeat API calls per N value.

Raw scored-block data is also saved to `data_cache/rerank_scored_blocks_31q.json` so notebook 14
(score-distribution/calibration check) can reuse it without a second pass.

In [1]:
import sys
import json
import time
from pathlib import Path

for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == "ModelPipeline":
        MODEL_ROOT = p
        break
if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))

from finrag_ml_tg1.loaders.ml_config_loader import MLConfig
from finrag_ml_tg1.rag_modules_src.synthesis_pipeline.supply_lines import (
    init_rag_components, run_supply_line_2_rag,
)
from finrag_ml_tg1.rag_modules_src.rag_pipeline.reranker import CohereReranker
from finrag_ml_tg1.rag_modules_src.utilities.retrieval_metrics import recall_at_k, reciprocal_rank

config = MLConfig()
rag = init_rag_components()            # reranker=None -- shared retrieval+expansion pass
rag.retriever.enable_variants = False   # in-memory only, matches notebooks 11/12

scorer = CohereReranker(
    retrieval_config={**config.cfg["retrieval"], "rerank_top_n_blocks": None, "rerank_min_score": 0.0},
    region=config.region, aws_access_key_id=config.aws_access_key,
    aws_secret_access_key=config.aws_secret_key,
)

GOLD_PATH = MODEL_ROOT.parent / "MLFlow_POC" / "data" / "p3_gold_test_suite_31q.json"
gold = json.loads(GOLD_PATH.read_text())
print(f"Loaded {len(gold)} gold questions")

[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
Loaded 31 gold questions


In [2]:
# One retrieval + one scoring pass per question. Save raw block data for reuse (notebook 14).
# Idempotent: if this has already run once (real Bedrock calls made), reuse the saved file
# rather than re-paying for retrieval + rerank on a re-run (e.g. after fixing a downstream bug).
_cache_path = MODEL_ROOT / "finrag_ml_tg1" / "data_cache" / "rerank_scored_blocks_31q.json"
if _cache_path.exists():
    per_question = json.loads(_cache_path.read_text())
    print(f"Loaded cached scored-block data for {len(per_question)} questions from {_cache_path.name} "
          f"(skipping real API calls -- delete this file to force a fresh run)")
else:
    per_question = []
    t_start = time.time()

    for i, gq in enumerate(gold, start=1):
        try:
            _, entities, bundle, unique_sents, _, _ = run_supply_line_2_rag(gq["question_text"], rag)
        except Exception as e:
            print(f"[{i:2d}/{len(gold)}] {gq['question_id']:12s} RETRIEVAL FAILED: {type(e).__name__}: {e}")
            continue

        blocks = scorer.score_blocks(gq["question_text"], unique_sents)
        if blocks is None:
            print(f"[{i:2d}/{len(gold)}] {gq['question_id']:12s} too few blocks to score, skipping")
            continue

        gold_ids = set(gq["evidence_sentence_ids"])
        block_records = []
        for b in blocks:
            sids = [str(sid) for sid in b.sentence_ids]
            block_records.append({
                "final_score": b.final_score,
                "base_score": b.base_score,
                "n_sentences": len(sids),
                "sentence_ids": sids,
                "is_gold_block": any(sid in gold_ids for sid in sids),
            })

        per_question.append({
            "question_id": gq["question_id"],
            "gold_version": gq.get("gold_version"),
            "retrieval_scope": gq.get("retrieval_scope"),
            "evidence_sentence_ids": list(gold_ids),
            "blocks": block_records,
        })
        n_gold_blocks = sum(1 for b in block_records if b["is_gold_block"])
        print(f"[{i:2d}/{len(gold)}] {gq['question_id']:12s} "
              f"{len(blocks)} blocks scored, {n_gold_blocks} contain gold evidence")

    elapsed = time.time() - t_start
    print(f"\n{len(per_question)}/{len(gold)} questions processed in {elapsed:.1f}s")

    SCORED_BLOCKS_PATH = MODEL_ROOT / "finrag_ml_tg1" / "data_cache" / "rerank_scored_blocks_31q.json"
    SCORED_BLOCKS_PATH.parent.mkdir(parents=True, exist_ok=True)
    SCORED_BLOCKS_PATH.write_text(json.dumps(per_question))
    print(f"Saved raw scored-block data -> {SCORED_BLOCKS_PATH}")

Loaded cached scored-block data for 31 questions from rerank_scored_blocks_31q.json (skipping real API calls -- delete this file to force a fresh run)


In [3]:
import polars as pl

def simulate_topn(per_question, top_n):
    """Simulate pruning to top_n blocks (by final_score desc, min_score=0.0 --
    matches the shipped reranker's own _select() logic exactly) and score recall/MRR."""
    rows = []
    for q in per_question:
        blocks_sorted = sorted(q["blocks"], key=lambda b: -b["final_score"])
        kept = blocks_sorted[:top_n] if top_n is not None else blocks_sorted
        kept_ids = [sid for b in kept for sid in b["sentence_ids"]]
        gold_ids = q["evidence_sentence_ids"]

        n_hits = sum(1 for sid in gold_ids if sid in kept_ids)
        gold_survived = (n_hits == len(gold_ids)) if gold_ids else True

        rr = reciprocal_rank(kept_ids, gold_ids)
        r5 = recall_at_k(kept_ids, gold_ids, 5)
        r30 = recall_at_k(kept_ids, gold_ids, 30)

        rows.append({
            "question_id": q["question_id"], "top_n": str(top_n) if top_n is not None else "all",
            "n_blocks_kept": len(kept), "n_sentences_kept": len(kept_ids),
            "gold_survived": gold_survived, "recall@5": r5, "recall@30": r30, "mrr": rr,
        })
    return rows

SWEEP_VALUES = [4, 8, 12, 16, 20, 24, 30, None]
sweep_rows = []
for n in SWEEP_VALUES:
    sweep_rows.extend(simulate_topn(per_question, n))

sweep_df = pl.DataFrame(sweep_rows)
summary = (
    sweep_df.group_by("top_n", maintain_order=True)
    .agg([
        pl.count().alias("n_questions"),
        pl.col("n_blocks_kept").mean().alias("avg_blocks_kept"),
        pl.col("n_sentences_kept").mean().alias("avg_sentences_kept"),
        pl.col("gold_survived").mean().alias("gold_survival_rate"),
        pl.col("recall@5").mean().alias("recall@5"),
        pl.col("recall@30").mean().alias("recall@30"),
        pl.col("mrr").mean().alias("mrr"),
    ])
)
# Preserve sweep order (group_by doesn't guarantee it across all polars versions)
order = {(str(n) if n is not None else "all"): i for i, n in enumerate(SWEEP_VALUES)}
summary = summary.sort(pl.col("top_n").map_elements(lambda x: order.get(x, 999), return_dtype=pl.Int64))
print(summary)

shape: (8, 8)
┌───────┬─────────────┬──────────────┬─────────────┬─────────────┬──────────┬───────────┬──────────┐
│ top_n ┆ n_questions ┆ avg_blocks_k ┆ avg_sentenc ┆ gold_surviv ┆ recall@5 ┆ recall@30 ┆ mrr      │
│ ---   ┆ ---         ┆ ept          ┆ es_kept     ┆ al_rate     ┆ ---      ┆ ---       ┆ ---      │
│ str   ┆ u32         ┆ ---          ┆ ---         ┆ ---         ┆ f64      ┆ f64       ┆ f64      │
│       ┆             ┆ f64          ┆ f64         ┆ f64         ┆          ┆           ┆          │
╞═══════╪═════════════╪══════════════╪═════════════╪═════════════╪══════════╪═══════════╪══════════╡
│ 4     ┆ 31          ┆ 4.0          ┆ 27.870968   ┆ 0.387097    ┆ 0.137097 ┆ 0.405914  ┆ 0.067993 │
│ 8     ┆ 31          ┆ 8.0          ┆ 50.903226   ┆ 0.548387    ┆ 0.137097 ┆ 0.47043   ┆ 0.073516 │
│ 12    ┆ 31          ┆ 12.0         ┆ 72.387097   ┆ 0.580645    ┆ 0.137097 ┆ 0.47043   ┆ 0.074161 │
│ 16    ┆ 31          ┆ 15.806452    ┆ 93.0        ┆ 0.580645    ┆ 0.137097 ┆

/var/folders/sb/x2_py571213939_gjl4zbhxc0000gn/T/ipykernel_51075/1319770343.py:36: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("n_questions"),


## Interpretation (fill in after running)

- Read `gold_survival_rate` first -- the safety endpoint. If it's already 1.0 (or very close)
  at N=8, going higher buys nothing on this dimension.
- Read `mrr`/`recall@5` next for whether a larger N changes ranking-quality metrics at all --
  given `ContextAssembler` discards order regardless, only `gold_survival_rate` and
  `n_sentences_kept` (the token-cost proxy) are actually architecture-relevant; `recall@5`/`mrr`
  here are diagnostic of the cross-encoder's own discrimination, not of what the LLM will see.
- If `gold_survival_rate` is flat across N=8 through N=30, that's evidence the project owner's
  top-15/16 hypothesis and the shipped top-8 default perform equivalently on this gold set --
  meaning the choice should be driven by cost (fewer sentences = fewer tokens) rather than by a
  quality argument that a sweep this size can't distinguish.
- Compare against `guidance/ANALYSIS_reranker_judgment_calls_2026-07-29.md` Sec 1.3's hard floor:
  N < 4 is guaranteed to fail multi-evidence questions requiring more than 4 distinct blocks --
  check whether N=4 in this sweep actually shows that predicted floor effect.